# Matcher: Export to OpenVINO (INT8) and Run on CPU
This notebook demonstrates the full workflow:
1. **Fit** a `Matcher` on a reference image + mask
2. **PyTorch baseline** with `Matcher.predict` (returns `list[Prediction]`)
3. **Export** the fitted model to an OpenVINO IR directory with `Matcher.to_openvino` (INT8 by default)
4. **Run inference** with `MatcherOpenVINO` on CPU
5. **Visualize** PyTorch vs OpenVINO predictions side-by-side
The reference features are baked into the exported IR, so `MatcherOpenVINO` does **not** support `fit()`.
To change the reference, re-run `Matcher.fit(...)` then `Matcher.to_openvino(...)`.

In [ ]:
from __future__ import annotations
from pathlib import Path
from time import time
import matplotlib.pyplot as plt
from instantlearn.data.base.sample import Category, Sample
from instantlearn.data.utils.image import read_image
from instantlearn.models import Matcher, MatcherOpenVINO
from instantlearn.models.torch_base import ExportConfig
from instantlearn.utils.constants import CompressionMode, SAMModelName
from instantlearn.visualizer import render_predictions, setup_colors

In [ ]:
EXAMPLES_DIR = Path("assets/coco")
EXPORT_DIR = Path("output/matcher_openvino")
REF_IMAGE = EXAMPLES_DIR / "000000286874.jpg"
REF_MASK = EXAMPLES_DIR / "000000286874_mask.png"
TARGET_IMAGES = [
    EXAMPLES_DIR / "000000390341.jpg",
    EXAMPLES_DIR / "000000173279.jpg",
    EXAMPLES_DIR / "000000267704.jpg",
]
TORCH_DEVICE = "cpu"  # Change to "cuda" or "xpu" if available
OV_DEVICE = "CPU"
CATEGORY = Category(0, "elephant")

## 1. Fit the Matcher model
We use the default `SAM-HQ-tiny` variant (fastest for PyTorch inference).
During OpenVINO export, the model automatically falls back to `SAM-HQ-base`
because SAM-HQ-tiny produces non-deterministic outputs as an OpenVINO IR.

In [ ]:
model = Matcher(
    device=TORCH_DEVICE,
    encoder_model="dinov3_small",
    sam=SAMModelName.SAM_HQ_TINY,
    confidence_threshold=0.25,  # Low threshold to show more predictions for visualization
)
ref_sample = Sample(
    image_path=str(REF_IMAGE),
    mask_paths=str(REF_MASK),
    categories=[CATEGORY],
)
model.fit(ref_sample)
print("Matcher fitted on reference image.")

## 2. PyTorch baseline predictions
Run the model in PyTorch to have a baseline for comparison. `predict()` returns
a list of `Prediction` objects (one per input image) with post-processing applied.

In [ ]:
target_samples = [Sample(image_path=str(p)) for p in TARGET_IMAGES]
pytorch_preds = model.predict(target_samples)
for i, pred in enumerate(pytorch_preds):
    print(f"  Target {i}: {pred.masks.shape[0]} masks, scores {pred.scores.tolist()}")

## 3. Export to OpenVINO
The export pipeline: PyTorch &rarr; ONNX &rarr; OpenVINO IR (`.xml` + `.bin`).
Reference features from `fit()` are frozen into the graph, so the exported
model takes only a target image as input. A `metadata.json` (input/patch size
and category id&rarr;name map) is written next to the IR so `MatcherOpenVINO`
can rebuild `Prediction.label_names` without re-fitting.
**Note:** Since we initialized with `SAM-HQ-tiny`, the export logs a warning and
automatically uses `SAM-HQ-base` for the exported model. INT4 compression is
rejected for Matcher (noisy masks); use INT8 or no compression.

In [ ]:
ir_dir = model.to_openvino(
    EXPORT_DIR,
    config=ExportConfig(compression=CompressionMode.INT8_SYM),
)
print(f"Exported IR directory: {ir_dir}")
print(f"Files: {[f.name for f in sorted(ir_dir.iterdir())]}")

## 4. Load and run the OpenVINO model
`MatcherOpenVINO(model_dir=...)` loads `model.xml` + `metadata.json`. The
reference is already baked in, so no `fit()` is needed. Pre-/post-processing
(resize, mask resize back to the frame) happen inside `predict()`, which returns
`Prediction` objects.

In [ ]:
ov_model = MatcherOpenVINO(model_dir=ir_dir, device=OV_DEVICE)
print(f"Model input size: {ov_model.input_size}")

In [ ]:
ov_preds = []
for img_path in TARGET_IMAGES:
    start = time()
    pred = ov_model.predict(Sample(image_path=str(img_path)))[0]
    elapsed = time() - start
    ov_preds.append(pred)
    print(f"  {img_path.name}: {pred.masks.shape[0]} masks, {elapsed:.3f}s, scores {pred.scores.tolist()}")

## 5. Visualize: PyTorch vs OpenVINO
Overlay predictions from both backends on the target images.

In [ ]:
colors = setup_colors({CATEGORY.id: CATEGORY.label})
target_images_np = [read_image(str(p)) for p in TARGET_IMAGES]
fig, axes = plt.subplots(len(TARGET_IMAGES), 2, figsize=(14, 7 * len(TARGET_IMAGES)))
for row, (img, pt_pred, ov_pred) in enumerate(zip(target_images_np, pytorch_preds, ov_preds, strict=False)):
    axes[row, 0].imshow(render_predictions(img, pt_pred, colors))
    axes[row, 0].set_title(f"PyTorch \u2014 {pt_pred.masks.shape[0]} masks", fontsize=13)
    axes[row, 0].axis("off")
    axes[row, 1].imshow(render_predictions(img, ov_pred, colors))
    axes[row, 1].set_title(f"OpenVINO ({OV_DEVICE}) \u2014 {ov_pred.masks.shape[0]} masks", fontsize=13)
    axes[row, 1].axis("off")
fig.suptitle("Matcher: PyTorch vs OpenVINO", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()